In [2]:
import torch
print(torch.cuda.is_available())
import pandas as pd
from autogluon.tabular import TabularPredictor

False


c:\Users\Hzaab\Desktop\MLSD project\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:


# -----------------------------
# Paths
# -----------------------------
TRAIN_PATH = r"C:\Users\Hzaab\Desktop\MLSD project\data\preprocessed\train.parquet"
TEST_PATH = r"C:\Users\Hzaab\Desktop\MLSD project\data\preprocessed\val.parquet"
TARGET = "fake"

# Where AutoGluon will save everything
AUTOML_PATH = r"C:\Users\Hzaab\Desktop\MLSD project\scratch\autogluon_auc"

# -----------------------------
# Load data
# -----------------------------
train_df = pd.read_parquet(TRAIN_PATH)
test_df = pd.read_parquet(TEST_PATH)

# -----------------------------
# Train AutoML
# -----------------------------
predictor = TabularPredictor(
    label=TARGET,
    eval_metric="roc_auc",
    path=AUTOML_PATH,
    verbosity=2,
).fit(
    train_data=train_df,
    presets="best_quality",
    time_limit=3600,   # 1 hour; change as needed
)

# -----------------------------
# Compare models
# -----------------------------
leaderboard = predictor.leaderboard(test_df, silent=True)
print("\n=== Leaderboard ===")
print(leaderboard)

# -----------------------------
# Evaluate on held-out test set
# -----------------------------
test_scores = predictor.evaluate(test_df)
print("\n=== Test metrics ===")
print(test_scores)

# -----------------------------
# Best model predictions
# -----------------------------
y_pred = predictor.predict(test_df.drop(columns=[TARGET]))
print("\nSample predictions:")
print(y_pred.head())

# Optional probabilities
y_prob = predictor.predict_proba(test_df.drop(columns=[TARGET]))
print("\nSample predicted probabilities:")
print(y_prob.head())

In [6]:
predictor.info()

{'path': 'C:\\Users\\Hzaab\\Desktop\\MLSD project\\scratch\\autogluon_auc',
 'label': 'fake',
 'random_state': 0,
 'version': '1.5.0',
 'features': ['profile pic',
  'nums/length username',
  'fullname words',
  'nums/length fullname',
  'name==username',
  'description length',
  'external URL',
  'private',
  '#posts',
  '#followers',
  '#follows'],
 'feature_metadata_in': <autogluon.common.features.feature_metadata.FeatureMetadata at 0x15ca873cad0>,
 'time_fit_preprocessing': 0.04372358322143555,
 'time_fit_training': 2696.409551858902,
 'time_fit_total': 2696.4532754421234,
 'time_limit': 2696.269781589508,
 'time_train_start': 1775735600.9189434,
 'num_rows_train': 2000,
 'num_cols_train': 11,
 'num_rows_val': None,
 'num_rows_test': None,
 'num_classes': 2,
 'problem_type': 'binary',
 'eval_metric': 'roc_auc',
 'best_model': 'WeightedEnsemble_L3',
 'best_model_score_val': np.float64(0.999371875),
 'best_model_stack_level': 3,
 'num_models_trained': 92,
 'num_bag_folds': 8,
 'max_

In [3]:

predictor = TabularPredictor.load(
    r"C:\Users\Hzaab\Desktop\MLSD project\scratch\autogluon_auc"   # or gpu path
)

In [ ]:
from sklearn.metrics import accuracy_score, f1_score, recall_score, precision_score

X_test = test_df.drop(columns=["fake"])
y_true = test_df["fake"]

# predictions
y_pred = predictor.predict(X_test)

results = {
    "accuracy": accuracy_score(y_true, y_pred),
    "f1": f1_score(y_true, y_pred),
    "recall": recall_score(y_true, y_pred),
    "precision": precision_score(y_true, y_pred),
}

Accuracy: 0.992
F1: 0.9795918367346939
Recall: 0.96
Precision: 1.0
